<a href="https://colab.research.google.com/github/weronikadomczewska/MCS-TimePoint-DTW/blob/main/timepoint/timepoint_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True

torch.backends.cudnn.benchmark = False

# Training on synthetic data

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [ ]:
import torch


def _patched_solve(B, A):
    return torch.linalg.solve(A, B), None


torch.solve = _patched_solve

from torch.utils.data import DataLoader

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
SIGNAL_TO_TRAIN = "abp"  # or "abp"
SIGNAL_GENERATION_METHOD = 'mader' # or "lognormal"

PATHS_MAPPING = {
    "mader": "/content/drive/MyDrive/dataset_synthetic_cpab/mader",
    "lognormal": "/content/drive/MyDrive/dataset_synthetic_cpab/lognormal",
}

In [ ]:
def collate_fn(batch):
    return {
        "x": torch.stack([b["x"] for b in batch]),
        "x_w": torch.stack([b["x_w"] for b in batch]),
        "kp": torch.stack([b["kp"] for b in batch]),
        "kp_w": torch.stack([b["kp_w"] for b in batch]),
        "match_mask": torch.stack([b["match_mask"] for b in batch]),
    }

In [ ]:
import torch
import numpy as np
import os

def kp_to_mask(kp_indices, length):
    mask = np.zeros(length, dtype=np.float32)
    kp_indices = kp_indices[(kp_indices >= 0) & (kp_indices < length)]
    mask[kp_indices] = 1.0
    return mask


def process_signal(x, x_w, kp, kp_w, window):
    start = 0

    x = x[:window]
    x_w = x_w[:window]

    kp = kp[(kp >= start) & (kp < start + window)] - start
    kp_w = kp_w[(kp_w >= start) & (kp_w < start + window)] - start

    kp_mask = kp_to_mask(kp, window)
    kp_w_mask = kp_to_mask(kp_w, window)

    return x, x_w, kp_mask, kp_w_mask


class NPZLoader(torch.utils.data.Dataset):
    def __init__(self, data_dir, use_signal="abp", window=1024):
        self.files = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith(".npz")]
        self.use_signal = use_signal
        self.window = window

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        data = np.load(self.files[idx])

        if self.use_signal == "abp":
            x = data["abp"]
            x_w = data["abp_warped"]
            kp = data["abp_kp"]
            kp_w = data["abp_kp_warped"]

        else:
            x = data["cbfv"]
            x_w = data["cbfv_warped"]
            kp = data["cbfv_kp"]
            kp_w = data["cbfv_kp_warped"]

        x, x_w, kp_mask, kp_w_mask = process_signal(x, x_w, kp, kp_w, self.window)

        match_mask = np.outer(kp_mask, kp_w_mask).astype(np.float32)

        return {
            "x": torch.tensor(x, dtype=torch.float32),
            "x_w": torch.tensor(x_w, dtype=torch.float32),
            "kp": torch.tensor(kp_mask),
            "kp_w": torch.tensor(kp_w_mask),
            "match_mask": torch.tensor(match_mask, dtype=torch.float32),
        }


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

In [ ]:
from torch.utils.data import Subset

# original dataset
dataset = NPZLoader(
    PATHS_MAPPING[SIGNAL_GENERATION_METHOD],
    use_signal=SIGNAL_TO_TRAIN
)

# choose subset size
subset_size = 1000

# first N samples
subset_indices = list(range(subset_size))

dataset = Subset(
    dataset,
    subset_indices
)

loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_fn
)

In [ ]:
batch = next(iter(loader))

x = batch["x"].unsqueeze(1)       # [B,1,L]
x_w = batch["x_w"].unsqueeze(1)

kp = batch["kp"]                  # [B,L]
kp_w = batch["kp_w"]
match_mask = batch["match_mask"]  # [B,L,L]

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class KeypointDecoder(nn.Module):
    """
    Detector Head for predicting keypoint logits.
    """

    def __init__(self, input_channels, cell_size=8):

        super().__init__()

        self.cell_size = cell_size

        self.conv = nn.Conv1d(
            input_channels,
            cell_size + 1,
            kernel_size=1
        )

    def forward(self, x):

        # x: [N, C, L/8]

        N, C, Lc = x.shape

        # ==========================================
        # raw logits
        # ==========================================

        x = self.conv(x)

        # ==========================================
        # remove dustbin
        # ==========================================

        x = x[:, :-1, :]

        # ==========================================
        # reshape to [N,1,L]
        # ==========================================

        x = x.permute(0, 2, 1).reshape(

            N,
            1,
            Lc * self.cell_size
        )

        return x


class DescriptorDecoder(nn.Module):
    """
    Descriptor Head for generating feature descriptors.
    """

    def __init__(self, input_channels, descriptor_dim=256):
        super().__init__()
        self.conv = nn.Conv1d(input_channels, descriptor_dim, kernel_size=1)
        self.upsample = nn.Upsample(scale_factor=8, mode="linear", align_corners=False)

    def forward(self, x):
        # x: [N, C, L/8]
        x = self.conv(x)  # [N, descriptor_dim, L/8]
        x = self.upsample(x)  # [N, descriptor_dim, L]
        # x = F.normalize(x, p=2, dim=1)  # L2 norm along channel dimension, now performed at loss.
        return x

In [ ]:
# https://github.com/BGU-CS-VIL/TimePoint/blob/main/TimePoint/models/wtconv1d.py

import torch
import torch.nn as nn
import torch.nn.functional as F

from functools import partial

import pywt


def create_wavelet_filter(wave, in_size, out_size, type=torch.float):
    w = pywt.Wavelet(wave)
    dec_hi = torch.tensor(w.dec_hi[::-1], dtype=type)
    dec_lo = torch.tensor(w.dec_lo[::-1], dtype=type)
    dec_filters = torch.stack([dec_lo, dec_hi], dim=0)

    dec_filters = dec_filters[:, None].repeat(in_size, 1, 1)

    rec_hi = torch.tensor(w.rec_hi[::-1], dtype=type).flip(dims=[0])
    rec_lo = torch.tensor(w.rec_lo[::-1], dtype=type).flip(dims=[0])
    rec_filters = torch.stack([rec_lo, rec_hi], dim=0)

    rec_filters = rec_filters[:, None].repeat(out_size, 1, 1)

    return dec_filters, rec_filters


def wavelet_transform(x, filters):
    b, c, l = x.shape
    pad = filters.shape[2] // 2 - 1
    x = F.conv1d(x, filters, stride=2, groups=c, padding=pad)
    x = x.reshape(b, c, 2, l // 2)
    return x


def inverse_wavelet_transform(x, filters):
    b, c, _, l_half = x.shape
    pad = filters.shape[2] // 2 - 1
    x = x.reshape(b, c * 2, l_half)
    x = F.conv_transpose1d(x, filters, stride=2, groups=c, padding=pad)
    return x


class WTConv1d(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size=5,
        stride=1,
        bias=True,
        wt_levels=1,
        wt_type="db1",
    ):
        super(WTConv1d, self).__init__()

        assert in_channels == out_channels

        self.in_channels = in_channels
        self.wt_levels = wt_levels
        self.stride = stride
        self.dilation = 1

        self.wt_filter, self.iwt_filter = create_wavelet_filter(
            wt_type, in_channels, in_channels, torch.float
        )
        self.wt_filter = nn.Parameter(self.wt_filter, requires_grad=False)
        self.iwt_filter = nn.Parameter(self.iwt_filter, requires_grad=False)

        self.wt_function = partial(wavelet_transform, filters=self.wt_filter)
        self.iwt_function = partial(inverse_wavelet_transform, filters=self.iwt_filter)

        self.base_conv = nn.Conv1d(
            in_channels,
            in_channels,
            kernel_size,
            padding="same",
            stride=1,
            dilation=1,
            groups=in_channels,
            bias=bias,
        )
        self.base_scale = _ScaleModule([1, in_channels, 1])

        self.wavelet_convs = nn.ModuleList(
            [
                nn.Conv1d(
                    in_channels * 2,
                    in_channels * 2,
                    kernel_size,
                    padding="same",
                    stride=1,
                    dilation=1,
                    groups=in_channels * 2,
                    bias=False,
                )
                for _ in range(self.wt_levels)
            ]
        )
        self.wavelet_scale = nn.ModuleList(
            [
                _ScaleModule([1, in_channels * 2, 1], init_scale=0.1)
                for _ in range(self.wt_levels)
            ]
        )

        if self.stride > 1:
            self.stride_filter = nn.Parameter(
                torch.ones(in_channels, 1, 1), requires_grad=False
            )
            self.do_stride = lambda x_in: F.conv1d(
                x_in,
                self.stride_filter,
                bias=None,
                stride=self.stride,
                groups=in_channels,
            )
        else:
            self.do_stride = None

    def forward(self, x):
        x_ll_in_levels = []
        x_h_in_levels = []
        shapes_in_levels = []

        curr_x_ll = x

        for i in range(self.wt_levels):
            curr_shape = curr_x_ll.shape
            shapes_in_levels.append(curr_shape)
            if curr_shape[2] % 2 > 0:
                curr_pads = (0, curr_shape[2] % 2)
                curr_x_ll = F.pad(curr_x_ll, curr_pads)

            curr_x = self.wt_function(curr_x_ll)
            curr_x_ll = curr_x[:, :, 0, :]

            shape_x = curr_x.shape
            curr_x_tag = curr_x.reshape(shape_x[0], shape_x[1] * 2, shape_x[3])
            curr_x_tag = self.wavelet_scale[i](self.wavelet_convs[i](curr_x_tag))
            curr_x_tag = curr_x_tag.reshape(shape_x)

            x_ll_in_levels.append(curr_x_tag[:, :, 0, :])
            x_h_in_levels.append(curr_x_tag[:, :, 1:2, :])

        next_x_ll = 0

        for i in range(self.wt_levels - 1, -1, -1):
            curr_x_ll = x_ll_in_levels.pop()
            curr_x_h = x_h_in_levels.pop()
            curr_shape = shapes_in_levels.pop()

            curr_x_ll = curr_x_ll + next_x_ll

            curr_x = torch.cat([curr_x_ll.unsqueeze(2), curr_x_h], dim=2)
            next_x_ll = self.iwt_function(curr_x)

            next_x_ll = next_x_ll[:, :, : curr_shape[2]]

        x_tag = next_x_ll
        assert len(x_ll_in_levels) == 0

        x = self.base_scale(self.base_conv(x))
        x = x + x_tag

        if self.do_stride is not None:
            x = self.do_stride(x)

        return x


class _ScaleModule(nn.Module):
    def __init__(self, dims, init_scale=1.0, init_bias=0):
        super(_ScaleModule, self).__init__()
        self.dims = dims
        self.weight = nn.Parameter(torch.ones(*dims) * init_scale)
        self.bias = None

    def forward(self, x):
        return torch.mul(self.weight, x)


In [ ]:
import torch.nn as nn

# https://github.com/BGU-CS-VIL/TimePoint/blob/main/TimePoint/models/layers.py


class ConvBlock1D(nn.Module):
    def __init__(
        self,
        c_in,
        c_out,
        kernel_size=3,
        stride=1,
        norm=nn.BatchNorm1d,
        act=nn.ReLU,
        padding=1,
    ):
        super(ConvBlock1D, self).__init__()
        self.layer = nn.Sequential(
            nn.Conv1d(
                c_in, c_out, kernel_size=kernel_size, stride=stride, padding=padding
            ),
            norm(c_out),
            act(inplace=True),
        )

    def forward(self, x):
        return self.layer(x)


class WTConvBlock1D(nn.Module):
    def __init__(
        self,
        c_in,
        c_out,
        kernel_size=3,
        stride=1,
        norm=nn.BatchNorm1d,
        act=nn.ReLU,
        wt_levels=3,
    ):
        super(WTConvBlock1D, self).__init__()
        self.layer = nn.Sequential(
            WTConv1d(
                c_in, c_in, kernel_size=kernel_size, wt_levels=wt_levels, stride=stride
            ),
            nn.Conv1d(c_in, c_out, kernel_size=1, stride=1, padding=0),
            norm(c_out),
            act(),
        )

    def forward(self, x):
        return self.layer(x)


In [ ]:
# import torch
import torch.nn as nn

# https://github.com/BGU-CS-VIL/TimePoint/blob/main/TimePoint/models/wtconv1d.py


class SharedEncoder(nn.Module):
    def __init__(
        self, input_channels=1, dims=[64, 64, 128, 128], stride=2, wt_levels=[3, 3, 3]
    ):
        super().__init__()
        self.stride = stride
        self.layer1 = ConvBlock1D(input_channels, dims[0], stride=1, padding="same")
        self.layer2 = WTConvBlock1D(
            dims[0], dims[1], stride=self.stride, wt_levels=wt_levels[0]
        )  # stride=2 to downsample
        self.layer3 = WTConvBlock1D(
            dims[1], dims[2], stride=self.stride, wt_levels=wt_levels[1]
        )
        self.layer4 = WTConvBlock1D(
            dims[2], dims[3], stride=self.stride, wt_levels=wt_levels[2]
        )

    def forward(self, x):
        # Input x: [N, C, L]
        x = self.layer1(x)  # [N, base_channels, L]
        x = self.layer2(x)  # [N, base_channels, L/2]
        x = self.layer3(x)  # [N, base_channels*2, L/4]
        x = self.layer4(x)  # [N, base_channels*2, L/8]
        return x  # Feature map of size L/8

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

def get_topk_in_original_order(X_desc, X_probas, K):
    """
    Get the descriptors of the top K keypoints from X_probas without changing their original order.

    Args:
        X_desc (torch.Tensor): The descriptors associated with keypoints, shape [N, C, L].
        X_probas (torch.Tensor): Tensor of keypoint probabilities, shape [N, L].
        K (int): Number of top elements to select per sample.

    Returns:
        X_topk (torch.Tensor): Tensor containing the descriptors of the top K keypoints per sample, shape [N, C, K].
    """
    N, C, L = X_desc.shape
    assert X_probas.shape == (N, L), "X_keypoints must have shape (N, L)"

    device = X_probas.device
    if K >= L:
        # If K is greater than or equal to L, return all points in original order
        # In this case, X_probas are the keypoint probabilities, and X_desc are the descriptors
        # We need to return sorted_topk_indices and X_topk in a consistent way.
        # If all points are kept, indices are just 0 to L-1
        sorted_topk_indices = torch.arange(L, device=device).unsqueeze(0).repeat(N, 1)
        return sorted_topk_indices, X_desc

    # Get the indices of the top K values per sample
    topk_values, topk_indices = torch.topk(X_probas, K, dim=1)
    # topk_indices: shape [N, K]

    # Sort the indices per sample to maintain original order
    sorted_topk_indices, _ = torch.sort(topk_indices, dim=1)
    # sorted_topk_indices: shape [N, K]

    # Expand indices for gathering
    indices_expanded = sorted_topk_indices.unsqueeze(1).expand(-1, C, -1)  # Shape: [N, C, K]

    # Gather descriptors along the L dimension (time steps)
    X_topk = torch.gather(X_desc, dim=2, index=indices_expanded)  # Shape: [N, C, K]

    return sorted_topk_indices, X_topk


def non_maximum_suppression(detection_prob, window_size=7):
    """
    Apply non-maximum suppression to the detection map.

    Args:
        detection_map: Tensor of shape [N, L].
        window_size: Size of the window for NMS.

    Returns:
        keypoints: Tensor of shape [N, L], boolean mask of keypoints after NMS.
    """
    if isinstance(detection_prob, np.ndarray):
        detection_prob = torch.from_numpy(detection_prob)

    N, L = detection_prob.shape

    # F.max_pool1d expects input of shape (N, C, L_in). Here C=1.
    detection_prob_3d = detection_prob.unsqueeze(1) # Shape becomes [N, 1, L]

    pooled, pooled_idx = F.max_pool1d(detection_prob_3d, kernel_size=window_size,
                                      stride=window_size, padding=window_size // 2,
                                      return_indices=True)

    # pooled_idx shape: [N, 1, L_pooled]. Squeeze the channel dimension.
    pooled_idx = pooled_idx.squeeze(1) # Shape [N, L_pooled]

    # Create a mask to zero out non-maximum values
    mask = torch.zeros_like(detection_prob, dtype=torch.bool)

    for i in range(N):
        # For each batch item, set True only at the indices that were maximum in their window
        mask[i, pooled_idx[i]] = True

    # Apply the mask to the original detection probabilities
    detection_prob_nms = detection_prob * mask.float()

    return detection_prob_nms

In [ ]:
import torch.nn as nn
import torch


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


class TimePointModel(nn.Module):
    def __init__(
        self, input_channels=1, encoder_dims=[64, 64, 128, 128], descriptor_dim=256
    ):
        super().__init__()

        self.encoder = SharedEncoder(input_channels, encoder_dims)

        encoder_output_channels = encoder_dims[-1]

        self.detector_head = KeypointDecoder(encoder_output_channels)
        self.descriptor_head = DescriptorDecoder(
            encoder_output_channels, descriptor_dim
        )

    def forward(self, x):
        N, C, L = x.shape

        features = self.encoder(x)

        # --- Keypoints ---
        S_scores = self.detector_head(features)  # [B,1,L]
        S_scores = S_scores[:, :, :L]
        S_scores = S_scores.squeeze(1)  # [B,L]

        # --- Descriptors ---
        D = self.descriptor_head(features)  # [B,D,L]
        D = D[:, :, :L]
        D = D.permute(0, 2, 1)  # [B,L,D]

        return S_scores, D

    def get_topk_points(self, x, kp_percent=1, nms_window=5):
        N, C, L = x.shape

        features = self.encoder(x)

        detection_proba = self.detector_head(features)[:, :, :L]
        descriptors = self.descriptor_head(features)[:, :, :L]

        detection_proba = detection_proba.squeeze(1)

        detection_proba = non_maximum_suppression(
            detection_proba, window_size=nms_window
        )

        if kp_percent < 1:
            num_kp = int(kp_percent * L)

            sorted_topk_indices, detection_proba = get_topk_in_original_order(
                detection_proba, detection_proba, K=num_kp
            )
        else:
            sorted_topk_indices = torch.arange(L)

        return sorted_topk_indices, detection_proba, descriptors

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# https://github.com/BGU-CS-VIL/TimePoint/blob/main/TimePoint/models/wtconv1d.py


class TimePointKeypointLoss(nn.Module):

    def __init__(self, pos_weight=80.0):

        super().__init__()

        self.pos_weight = pos_weight

    def forward(self, S_logits, Y_true):

        Y_true = Y_true.float()

        pos_weight = torch.tensor(

            self.pos_weight,

            device=S_logits.device
        )

        loss_fn = nn.BCEWithLogitsLoss(

            pos_weight=pos_weight
        )

        loss = loss_fn(

            S_logits,

            Y_true
        )

        return loss


class TimePointDescriptorLoss(nn.Module):
    def __init__(self, mp=1.0, mn=0.1):
        """
        mp: positive margin (target similarity)
        mn: negative margin (maximum allowed similarity)
        """
        super().__init__()
        self.mp = mp
        self.mn = mn

    def forward(self, D, D_prime, match_mask):
        """
        D:         [B, N, D]  - descriptors (original)
        D_prime:   [B, N, D]  - descriptors (warped)
        match_mask:[B, N, N]  - 1 if (i,j) is matching pair

        returns: scalar loss
        """

        # normalize descriptors
        D = F.normalize(D, p=2, dim=-1)
        D_prime = F.normalize(D_prime, p=2, dim=-1)

        # cosine similarity matrix
        # shape: [B, N, N]
        sim = torch.bmm(D, D_prime.transpose(1, 2))

        # positive loss
        pos_loss = match_mask * (F.relu(self.mp - sim) ** 2)

        # negative loss
        neg_mask = 1.0 - match_mask
        neg_loss = neg_mask * (F.relu(sim - self.mn) ** 2)

        # combine
        loss = pos_loss + neg_loss

        return loss.mean()

In [ ]:
class TimePointOverallLoss(nn.Module):
    def __init__(self, mp=1.0, mn=0.1, lambda_desc=1.0):
        super().__init__()

        self.kp_loss_fn = TimePointKeypointLoss()
        self.desc_loss_fn = TimePointDescriptorLoss(mp=mp, mn=mn)

        self.lambda_desc = lambda_desc

    def forward(
        self,
        S_logits,
        Y_true,
        S_prime_logits,
        Y_prime_true,
        D,
        D_prime,
        match_mask,
    ):
        # Keypoint losses (with sigmoid inside)
        loss_kp_orig = self.kp_loss_fn(S_logits, Y_true)
        loss_kp_warped = self.kp_loss_fn(S_prime_logits, Y_prime_true)

        # Descriptor loss
        loss_desc = self.desc_loss_fn(D, D_prime, match_mask)

        # Total loss
        total_loss = loss_kp_orig + loss_kp_warped + self.lambda_desc * loss_desc

        return total_loss, {
            "total": total_loss.item(),
            "kp_orig": loss_kp_orig.item(),
            "kp_warped": loss_kp_warped.item(),
            "desc": loss_desc.item(),
        }

In [ ]:
model = TimePointModel()
model = model.to(device)

criterion = TimePointOverallLoss(
    mp=1.0,
    mn=0.1,
    lambda_desc=1.0
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
S, D = model(x.to(device))
S_w, D_w = model(x_w.to(device))

print("S:", S.shape)
print("D:", D.shape)
print("match_mask:", match_mask.shape)

In [ ]:
import os

save_dir = f"/content/drive/MyDrive/timepoint_weights_{SIGNAL_TO_TRAIN}_{SIGNAL_GENERATION_METHOD}"
os.makedirs(save_dir, exist_ok=True)

os.makedirs(f"{save_dir}/training_logs", exist_ok=True)

In [ ]:
from tqdm.auto import tqdm
import pandas as pd

epochs = 50

loss_history = []

for epoch in range(epochs):

    print(f"\nEpoch {epoch}")

    model.train()

    epoch_loss_dict = {

        "total": 0.0,
        "kp_orig": 0.0,
        "kp_warped": 0.0,
        "desc": 0.0,
    }

    total_loss = 0.0

    for batch in tqdm(loader, desc=f"Epoch {epoch}"):

        # ==========================================
        # data
        # ==========================================

        x = batch["x"].float().to(device).unsqueeze(1)

        x_w = batch["x_w"].float().to(device).unsqueeze(1)

        kp = batch["kp"].float().to(device)

        kp_w = batch["kp_w"].float().to(device)

        match_mask = batch["match_mask"].float().to(device)

        # ==========================================
        # forward
        # ==========================================

        S_logits, D = model(x)

        S_w_logits, D_w = model(x_w)

        # ==========================================
        # loss
        # ==========================================

        loss, loss_dict = criterion(

            S_logits,
            kp,

            S_w_logits,
            kp_w,

            D,
            D_w,

            match_mask
        )

        # ==========================================
        # backward
        # ==========================================

        optimizer.zero_grad()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        # ==========================================
        # accumulate losses
        # ==========================================

        total_loss += loss.item()

        for key in epoch_loss_dict:

            epoch_loss_dict[key] += loss_dict[key]

    # ==============================================
    # average losses
    # ==============================================

    num_batches = len(loader)

    for key in epoch_loss_dict:

        epoch_loss_dict[key] /= num_batches

    loss_history.append(
        epoch_loss_dict
    )

    print(epoch_loss_dict)

    loss_df = pd.DataFrame(loss_history)

    loss_df.to_csv(
        f"{save_dir}/training_logs/loss_history.csv",
        index=False
    )


    save_path = os.path.join(

        save_dir,

        f"model_epoch_{epoch}.pth"
    )

    torch.save({

        "epoch": epoch,

        "model_state_dict": model.state_dict(),

        "optimizer_state_dict": optimizer.state_dict(),

        "loss": epoch_loss_dict["total"],

    }, save_path)

In [ ]:
def visualize(sample, model):
    model.eval()

    x = sample["x"].unsqueeze(0).unsqueeze(0).to(device)
    x_w = sample["x_w"].unsqueeze(0).unsqueeze(0).to(device)

    with torch.no_grad():
        S_logits, _ = model(x)
        S_w_logits, _ = model(x_w)

    S = torch.sigmoid(S_logits).cpu().numpy()[0]
    S_w = torch.sigmoid(S_w_logits).cpu().numpy()[0]

    t = np.arange(len(S))

    plt.figure(figsize=(12,5))
    plt.plot(t, sample["x"], label=SIGNAL_TO_TRAIN.upper())
    plt.plot(t, sample["x_w"], label="Warped", alpha=0.7)

    plt.scatter(t[S > 0.5], sample["x"][S > 0.5], color="red", label="KP")
    plt.scatter(t[S_w > 0.5], sample["x_w"][S_w > 0.5], color="blue", label="KP warped")

    plt.legend()
    plt.title("Predicted keypoints")
    plt.show()

In [ ]:
# sample = dataset[0]
# visualize(sample, model)

### Training Loss Evolution

Let's visualize the training loss over the epochs to observe the model's learning progress.

In [ ]:
import pandas as pd

loss_path = "/content/drive/MyDrive/timepoint_weights_abp_mader/training_logs/loss_history.csv"
loss_df = pd.read_csv(loss_path)

# Convert to list of dicts in case other cells expect 'loss_history'
loss_history = loss_df.to_dict('records')

loss_df.head()

In [ ]:
import torch
import os

# Znajdź epokę z najmniejszą wartością 'total' loss
best_synth_epoch = loss_df['total'].idxmin()
best_synth_loss = loss_df['total'].min()

print(f"Best synthetic model is from epoch: {best_synth_epoch}")
print(f"Lowest total loss value: {best_synth_loss:.4f}")

# Zbuduj ścieżkę do modelu z tej epoki
best_synth_model_path = os.path.join(save_dir, f"model_epoch_{best_synth_epoch}.pth")
print(f"Best model path: {best_synth_model_path}")

# Załaduj najlepszy model do zmiennej (używanej w późniejszych komórkach)
if os.path.exists(best_synth_model_path):
    best_checkpoint_pre_fine_tuning = torch.load(best_synth_model_path, map_location=device)
    print("Best synthetic model checkpoint loaded successfully.")
else:
    print(f"Checkpoint file not found: {best_synth_model_path}")


In [ ]:
# import matplotlib.pyplot as plt
# import pandas as pd

# # Convert loss_history to a DataFrame for easier plotting
# loss_df = pd.DataFrame(loss_history)

# plt.figure(figsize=(12, 6))
# plt.plot(loss_df.index, loss_df['total'], label='Total Loss')
# plt.plot(loss_df.index, loss_df['kp_orig'], label='Keypoint Original Loss')
# plt.plot(loss_df.index, loss_df['kp_warped'], label='Keypoint Warped Loss')
# plt.plot(loss_df.index, loss_df['desc'], label='Descriptor Loss')

# plt.xlabel('Epoch')
# plt.ylabel('Loss function value')
# plt.title(f'Overall loss value - ABP')
# plt.legend()
# plt.grid(True)
# plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# LOAD SYNTHETIC TRAINING LOG
# ============================================================

loss_df = pd.read_csv(
    loss_path
    # f"{save_dir}/training_logs/loss_history.csv"
)

# ============================================================
# PLOT TRAINING LOSSES
# ============================================================

fig, axes = plt.subplots(

    2,

    2,

    figsize=(16,10)
)

axes = axes.flatten()

loss_types = [

    "total",

    "kp_orig",

    "kp_warped",

    "desc",
]

titles = [

    "Total Loss",

    "Keypoint Loss (Original)",

    "Keypoint Loss (Warped)",

    "Descriptor Loss",
]

for ax, loss_name, title in zip(

    axes,

    loss_types,

    titles
):

    ax.plot(

        loss_df[loss_name],

        linewidth=2
    )

    ax.set_title(title)

    ax.set_xlabel("Epoch")

    ax.set_ylabel("Loss")

    ax.grid()

plt.suptitle(

    "Synthetic training loss evolution - CBFV-M",

    fontsize=16
)

plt.tight_layout()

plt.show()

Selection of model with the lowest overall loss

# Fine-tuning on real data

Adding both baseline and position change data, excluding patients with signal issues discivered during EDA analysis

In [ ]:
import os
from torch.utils.data import ConcatDataset

# Lista plików pacjentów do wykluczenia
exclude_files = [
    "sample_PAC_02.npz",
    "sample_PAC_03.npz",
    "sample_PAC_33.npz",
    "sample_PAC_38.npz",
    "sample_PAC_39.npz"
]

baseline_dataset = NPZLoader(
    "/content/drive/MyDrive/data_for_finetuning/baseline",
    use_signal=SIGNAL_TO_TRAIN,
    window=1024,
)
# Wykluczenie plików
baseline_dataset.files = [f for f in baseline_dataset.files if os.path.basename(f) not in exclude_files]

position_dataset = NPZLoader(
    "/content/drive/MyDrive/data_for_finetuning/position",
    use_signal=SIGNAL_TO_TRAIN,
    window=1024,
)
# Wykluczenie plików
position_dataset.files = [f for f in position_dataset.files if os.path.basename(f) not in exclude_files]

dataset = ConcatDataset([
    baseline_dataset,
    position_dataset,
])

In [ ]:
print("Baseline:", len(baseline_dataset))
print("Position :", len(position_dataset))
print("Total    :", len(dataset))

In [ ]:
import os
import torch
import pandas as pd
from torch.utils.data import DataLoader

epochs_finetune = 10
synthetic_epochs = 50

# Re-initialize the loader with the new ConcatDataset
loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_fn
)

finetuned_base_dir = f"{save_dir}_fine_tuned"
os.makedirs(finetuned_base_dir, exist_ok=True)

for synth_epoch in range(synthetic_epochs):
    print(f"\n{'='*50}\nFine-tuning synthetic model from epoch {synth_epoch}\n{'='*50}")

    # 1. Load the corresponding synthetic model
    model = TimePointModel().to(device)
    synth_model_path = os.path.join(save_dir, f"model_epoch_{synth_epoch}.pth")

    if not os.path.exists(synth_model_path):
        print(f"Checkpoint not found: {synth_model_path}. Skipping.")
        continue

    checkpoint = torch.load(synth_model_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])

    # 2. Freeze the encoder
    for param in model.encoder.parameters():
        param.requires_grad = False

    # 3. Re-initialize the optimizer for the fine-tuning phase (only unfreezed params)
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=1e-4
    )

    loss_history_finetune = [] # History for this specific model

    # 4. Fine-tuning loop
    for epoch in range(epochs_finetune):
        model.train()
        total_loss = 0

        epoch_loss_dict = {
            "total": 0.0,
            "kp_orig": 0.0,
            "kp_warped": 0.0,
            "desc": 0.0,
        }

        for batch in loader:
            x = batch["x"].to(device).unsqueeze(1)     # [B,1,L]
            x_w = batch["x_w"].to(device).unsqueeze(1)

            kp = batch["kp"].to(device)
            kp_w = batch["kp_w"].to(device)

            match_mask = batch["match_mask"].to(device)

            # forward
            S_logits, D = model(x)
            S_w_logits, D_w = model(x_w)

            # loss
            loss, loss_dict = criterion(
                S_logits, kp,
                S_w_logits, kp_w,
                D, D_w,
                match_mask
            )

            # backward
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            for key in epoch_loss_dict:
                epoch_loss_dict[key] += loss_dict[key]

        # Average losses over batches
        num_batches = len(loader)
        for key in epoch_loss_dict:
            epoch_loss_dict[key] /= num_batches

        # Store loss_dict for the current epoch
        loss_history_finetune.append(epoch_loss_dict)

        print(f"  FT Epoch {epoch}/{epochs_finetune - 1} - Total loss: {epoch_loss_dict['total']:.4f}")

    # 5. Save the final fine-tuned model for this synthetic epoch
    final_save_dir = os.path.join(finetuned_base_dir, f"model_epoch_{synth_epoch}")
    os.makedirs(final_save_dir, exist_ok=True)

    loss_df = pd.DataFrame(loss_history_finetune)
    loss_df.to_csv(os.path.join(final_save_dir, "loss_history.csv"), index=False)

    final_save_path = os.path.join(final_save_dir, "model_final.pth")

    torch.save(
        {
            "synthetic_epoch": synth_epoch,
            "fine_tune_epochs": epochs_finetune,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss": epoch_loss_dict['total'],
        },
        final_save_path,
    )

    print(f"Saved final fine-tuned model and logs to: {final_save_dir}")

In [ ]:
import os
import torch
import matplotlib.pyplot as plt

finetuned_base_dir = f"/content/drive/MyDrive/timepoint_weights_abp_mader_fine_tuned"
losses = []
epochs_list = []
best_loss = float('inf')


best_epoch = -1
best_model_path = ""
best_synth_epoch = -1

# Gather loss data from all fine-tuned checkpoints
for synth_epoch in range(50):
    model_path = os.path.join(finetuned_base_dir, f"model_epoch_{synth_epoch}", "model_final.pth")
    if os.path.exists(model_path):
        checkpoint = torch.load(model_path, map_location=device)
        loss = checkpoint.get("loss", float('inf'))
        losses.append(loss)
        epochs_list.append(synth_epoch)

        # Track the best model
        if loss < best_loss:
            best_loss = loss
            best_epoch = synth_epoch
            best_model_path = model_path
            best_synth_epoch = synth_epoch

# 2. Select the best model
print(f"Best fine-tuned model is from synthetic epoch: {best_epoch}")
print(f"Lowest total loss value: {best_loss:.4f}")
print(f"Best model path: {best_model_path}")
print(f"Best synthetic epoch: {best_synth_epoch}")

# Load the best model into the 'model' variable
if best_model_path:
    model = TimePointModel().to(device)
    best_checkpoint = torch.load(best_model_path, map_location=device)
    model.load_state_dict(best_checkpoint["model_state_dict"])
    model.eval()
    print("Best fine-tuned model loaded successfully.")
else:
    print("No fine-tuned models found.")

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.flatten()

loss_keys = ['total', 'kp_orig', 'kp_warped', 'desc']
titles = ['Total Loss', 'Keypoint Original Loss', 'Keypoint Warped Loss', 'Descriptor Loss']

# Loop through all 50 synthetic epochs and plot each loss type on its respective subplot
for synth_epoch in range(50):
    csv_path = os.path.join(finetuned_base_dir, f"model_epoch_{synth_epoch}", "loss_history.csv")
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        for i, key in enumerate(loss_keys):
            axes[i].plot(df.index, df[key], label=f'Model {synth_epoch}')

for i, ax in enumerate(axes):
    ax.set_title(titles[i])
    ax.set_xlabel('Fine-tuning Epoch')
    ax.set_ylabel('Loss')
    ax.grid(True, alpha=0.3)

# Place the legend outside the top-right subplot (axes[1]) in 2 columns
axes[1].legend(ncol=2, bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small')

plt.suptitle('Fine-tuning losses evolution - ABP', fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
# Visualize keypoints before fine-tuning (using the checkpoint loaded in cell 15)
print("Visualizing keypoints BEFORE fine-tuning...")

# Create a new model instance to load the pre-fine-tuned weights
model_before_finetuning = TimePointModel().to(device)
checkpoint_before_finetuning = best_checkpoint_pre_fine_tuning
model_before_finetuning.load_state_dict(checkpoint_before_finetuning["model_state_dict"])
model_before_finetuning.eval()

# Use the same sample as before
sample_to_visualize = dataset[0] # Use the first sample from the fine-tuning dataset
visualize(sample_to_visualize, model_before_finetuning)


In [ ]:
# Visualize keypoints AFTER fine-tuning (using the fine-tuned best synthetic model)
print("Visualizing keypoints AFTER fine-tuning (for the best synthetic model)...")

import os
import torch

model_after_finetuning = TimePointModel().to(device)

# Path to the fine-tuned version of the best synthetic model
finetuned_best_synth_path = os.path.join(finetuned_base_dir, f"model_epoch_{best_synth_epoch}", "model_final.pth")

print(f"Loading fine-tuned model from: {finetuned_best_synth_path}")
checkpoint_after_finetuning = torch.load(finetuned_best_synth_path, map_location=device)
model_after_finetuning.load_state_dict(checkpoint_after_finetuning["model_state_dict"])
model_after_finetuning.eval()

# Use the same sample as before
sample_to_visualize = dataset[0] # Use the first sample from the fine-tuning dataset
visualize(sample_to_visualize, model_after_finetuning)


In [ ]:
import matplotlib.pyplot as plt
import torch
import numpy as np

def visualize_comparison(sample, model_before, model_after):
    model_before.eval()
    model_after.eval()

    x = sample["x"].unsqueeze(0).unsqueeze(0).to(device)
    x_w = sample["x_w"].unsqueeze(0).unsqueeze(0).to(device)

    with torch.no_grad():
        S_logits_b, _ = model_before(x)
        S_w_logits_b, _ = model_before(x_w)
        S_logits_a, _ = model_after(x)
        S_w_logits_a, _ = model_after(x_w)

    S_b = torch.sigmoid(S_logits_b).cpu().numpy()[0]
    S_w_b = torch.sigmoid(S_w_logits_b).cpu().numpy()[0]
    S_a = torch.sigmoid(S_logits_a).cpu().numpy()[0]
    S_w_a = torch.sigmoid(S_w_logits_a).cpu().numpy()[0]

    t = np.arange(len(S_b))

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

    # Plot BEFORE fine-tuning
    ax1.plot(t, sample["x"], label=SIGNAL_TO_TRAIN.upper())
    ax1.plot(t, sample["x_w"], label="Warped", alpha=0.7)
    ax1.scatter(t[S_b > 0.5], sample["x"][S_b > 0.5], color="red", label="KP (Before)")
    ax1.scatter(t[S_w_b > 0.5], sample["x_w"][S_w_b > 0.5], color="blue", label="KP warped (Before)")
    ax1.set_title("Predicted keypoints before fine-tuning - ABP")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Plot AFTER fine-tuning
    ax2.plot(t, sample["x"], label=SIGNAL_TO_TRAIN.upper())
    ax2.plot(t, sample["x_w"], label="Warped", alpha=0.7)
    ax2.scatter(t[S_a > 0.5], sample["x"][S_a > 0.5], color="red", label="KP (After)")
    ax2.scatter(t[S_w_a > 0.5], sample["x_w"][S_w_a > 0.5], color="blue", label="KP warped (After)")
    ax2.set_title("Predicted keypoints after fine-tuning - ABP")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# Run the comparison side-by-side
print("Visualizing comparison on a single canvas...")
visualize_comparison(sample_to_visualize, model_before_finetuning, model_after_finetuning)


##Experiments

Number of keypoints and number of epochs

In [ ]:
from scipy.stats import zscore

def normalize_signal(x):

    x = np.asarray(x)

    return zscore(x)

In [ ]:
def count_keypoints_for_model(

    checkpoint_path,
    dataset_paths,
    model_class,
    device,
):

    # ============================================
    # load model
    # ============================================

    model = model_class().to(device)

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    model.eval()

    # ============================================
    # stats
    # ============================================

    kp_counts = []

    # ============================================
    # iterate files
    # ============================================

    for path in dataset_paths:

        data = np.load(path)

        signal = data["cbfv"]

        # ========================================
        # sparse representation
        # ========================================

        kp_idx, desc, probs = (
            extract_sparse_representation(

                signal,

                model,

                device,


                nms_window=15,
            )
        )

        kp_counts.append(
            len(kp_idx)
        )

    return {

        "mean_kp": np.mean(kp_counts),

        "std_kp": np.std(kp_counts),

        "all_kp": kp_counts,
    }

In [ ]:
@torch.no_grad()
def extract_sparse_representation(
    signal,
    model,
    device,
    threshold=0.3,
    nms_window=15,
):

    # ==========================================
    # normalize
    # ==========================================

    signal = normalize_signal(signal)

    x = torch.tensor(
        signal,
        dtype=torch.float32
    ).unsqueeze(0).unsqueeze(0).to(device)

    # ==========================================
    # forward
    # ==========================================

    S_logits, D = model(x)

    # logits -> probabilities
    S_probs = torch.sigmoid(S_logits)

    # [B,L]
    if S_probs.ndim == 3:
        S_probs = S_probs.squeeze(1)

    # ==========================================
    # NMS
    # ==========================================

    S_nms = non_maximum_suppression(

        S_probs.clone(),

        window_size=nms_window
    )

    # ==========================================
    # thresholding
    # ==========================================

    kp_idx = torch.where(

        S_nms.squeeze(0) > threshold

    )[0]

    # ==========================================
    # descriptors
    # ==========================================

    # D: [B,L,D]
    # -> [B,D,L]
    D_for_gather = D.permute(0, 2, 1)

    # select descriptors at kp positions
    desc_topk = D_for_gather[
        0,
        :,
        kp_idx
    ]

    # ==========================================
    # numpy conversion
    # ==========================================

    kp_idx = kp_idx.cpu().numpy()

    desc_topk = (
        desc_topk
        .T
        .cpu()
        .numpy()
    )

    probs = (
        S_probs
        .squeeze(0)
        .cpu()
        .numpy()
    )

    return kp_idx, desc_topk, probs

In [ ]:
baseline_dir = "/content/drive/MyDrive/data_for_finetuning/baseline"

position_dir = "/content/drive/MyDrive/data_for_finetuning/position"

In [ ]:
baseline_files = [

    os.path.join(
        baseline_dir,
        f
    )

    for f in os.listdir(baseline_dir)

    if f.endswith(".npz") and f not in exclude_files
]

position_files = [

    os.path.join(
        position_dir,
        f
    )

    for f in os.listdir(position_dir)

    if f.endswith(".npz") and f not in exclude_files
]

all_files = baseline_files + position_files

## DTW testing

In [ ]:
%pip install fastdtw

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

from fastdtw import fastdtw
from scipy.spatial.distance import cosine, euclidean
from scipy.stats import zscore

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch

# Znajdź plik pacjenta 15
patient_file = next((f for f in all_files if "PAC_15" in f), None)
if patient_file is None:
    print("Nie znaleziono pliku dla pacjenta 15. Używam pierwszego dostępnego pliku.")
    patient_file = all_files[0]

print(f"Używany plik: {patient_file}")
data = np.load(patient_file)

# Pobierz 1-minutowy segment (12000 próbek) do testów, aby nie trwało to zbyt długo
segment_len = 12000
abp_seg = data["abp"][:segment_len]
cbfv_seg = data["cbfv"][:segment_len]

epochs_list = []
kp_counts_abp = []
kp_counts_cbfv = []

model = TimePointModel().to(device)
print("Rozpoczynam ekstrakcję punktów dla kolejnych epok...")

for epoch in range(50):
    model_path = os.path.join(finetuned_base_dir, f"model_epoch_{epoch}", "model_final.pth")
    if not os.path.exists(model_path):
        continue

    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    # Ekstrakcja
    kp_abp, desc_abp, _ = extract_sparse_representation(abp_seg, model, device)
    kp_cbfv, desc_cbfv, _ = extract_sparse_representation(cbfv_seg, model, device)

    kp_counts_abp.append(len(kp_abp))
    kp_counts_cbfv.append(len(kp_cbfv))
    epochs_list.append(epoch)

# Plot
fig, ax = plt.subplots(figsize=(12, 6))

ax.set_xlabel('Pre-training epoch')
ax.set_ylabel('Number of keypoints')
ax.plot(epochs_list, kp_counts_cbfv, color='tab:blue', marker='x', linestyle='--', label='KP (CBFV)')
ax.plot(epochs_list, kp_counts_abp, color='tab:orange', marker='o', linestyle='-', label='KP (ABP)')

fig.tight_layout()
plt.title("Keypoint detection statistics depending on number of pre-training epochs")
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean, cosine

# Find file for patient 15
patient_file = next((f for f in all_files if "PAC_15" in f), None)
if patient_file is None:
    print("Nie znaleziono pliku dla pacjenta 15. Używam pierwszego dostępnego pliku.")
    patient_file = all_files[0]

print(f"Używany plik: {patient_file}")
data = np.load(patient_file)

# Load FULL signals and flatten to 1-D
abp_full = data["abp"].flatten()
cbfv_full = data["cbfv"].flatten()
print(f"Długość pełnego sygnału: {len(abp_full)} próbek")

# 1. Full signal DTW
print("\nRozpoczynam obliczanie DTW na pełnych sygnałach (to może potrwać kilka/kilkanaście sekund)...")
start_time = time.time()
# Use reshape(-1, 1) so scipy euclidean receives 1-D vectors instead of scalars
distance_full, _ = fastdtw(abp_full.reshape(-1, 1), cbfv_full.reshape(-1, 1), dist=euclidean)
full_dtw_time = time.time() - start_time
print(f"Czas DTW (Pełny sygnał): {full_dtw_time:.4f} s")

# 2. Keypoints (Sparse) DTW
print("\nRozpoczynam ekstrakcję punktów kluczowych i obliczanie DTW...")
model.eval()
# Extract from full signals at once
kp_abp, desc_abp, _ = extract_sparse_representation(abp_full, model, device)
kp_cbfv, desc_cbfv, _ = extract_sparse_representation(cbfv_full, model, device)

print(f"Wykryto punktów kluczowych - ABP: {len(kp_abp)}, CBFV: {len(kp_cbfv)}")

start_time = time.time()
if len(desc_abp) > 0 and len(desc_cbfv) > 0:
    distance_sparse, _ = fastdtw(desc_abp, desc_cbfv, dist=cosine)
else:
    distance_sparse = 0.0
sparse_dtw_time = time.time() - start_time
print(f"Czas DTW (Punkty kluczowe): {sparse_dtw_time:.4f} s")

speedup = full_dtw_time / sparse_dtw_time if sparse_dtw_time > 0 else float('inf')
print(f"\nPrzyspieszenie dzięki punktom kluczowym: {speedup:.2f}x")

# 3. Comparison plot
fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.bar(['Full Signal (Full DTW)', 'Keypoints (Sparse DTW)'],
              [full_dtw_time, sparse_dtw_time],
              color=['tab:red', 'tab:green'],
              alpha=0.8)

ax.set_ylabel('Execution time [s]')
ax.set_title(f'DTW execution time comparison for signal of length {len(abp_full)} samples - ABP')

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + (0.02 * max(full_dtw_time, sparse_dtw_time)),
            f"{yval:.4f} s", ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
import time
import random
import numpy as np
import pandas as pd
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean, cosine

# Ustawienie globalnego seeda (powinno być już ustawione wyżej, ale dla pewności)
random.seed(SEED)
np.random.seed(SEED)

# Losowy wybór 10 sygnałów
num_samples = 10
selected_files = random.sample(all_files, min(num_samples, len(all_files)))

full_dtw_times = []
sparse_dtw_times = []

print(f"Rozpoczynam testy DTW na {len(selected_files)} losowych sygnałach...")
model.eval()

for i, file_path in enumerate(selected_files):
    data = np.load(file_path)
    abp_full = data["abp"].flatten()
    cbfv_full = data["cbfv"].flatten()

    # 1. Full signal DTW
    start_time = time.time()
    _, _ = fastdtw(abp_full.reshape(-1, 1), cbfv_full.reshape(-1, 1), dist=euclidean)
    full_time = time.time() - start_time
    full_dtw_times.append(full_time)

    # 2. Keypoints (Sparse) DTW
    kp_abp, desc_abp, _ = extract_sparse_representation(abp_full, model, device)
    kp_cbfv, desc_cbfv, _ = extract_sparse_representation(cbfv_full, model, device)

    start_time = time.time()
    if len(desc_abp) > 0 and len(desc_cbfv) > 0:
        _, _ = fastdtw(desc_abp, desc_cbfv, dist=cosine)
    sparse_time = time.time() - start_time
    sparse_dtw_times.append(sparse_time)

    print(f"Sygnał {i+1}/{len(selected_files)}: Full = {full_time:.4f}s | Sparse = {sparse_time:.4f}s")

# Tworzenie tabeli wyników
results_df = pd.DataFrame({
    "Metoda": ["Full Signal DTW", "Keypoints (Sparse) DTW"],
    "Średni czas [s]": [np.mean(full_dtw_times), np.mean(sparse_dtw_times)],
    "Odchylenie standardowe [s]": [np.std(full_dtw_times), np.std(sparse_dtw_times)]
})

print("\n--- Podsumowanie wyników ---")
display(results_df)


## Eksperyment: Klastrowanie (Mader vs Lognormal dla CBFV)
Porównanie zdolności separacji faz (baseline vs zmiana pozycji) przy użyciu K-Means na cechach wyekstrahowanych z modeli pretrenowanych różnymi metodami.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

In [ ]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

def evaluate_physio_clustering(model_path, baseline_files, position_files, device, segment_len=12000, n_clusters=2):
    print(f"Ocenianie klastrowania na nowych cechach fizjologicznych dla: {model_path}")

    # Ładowanie modelu
    model = TimePointModel().to(device)
    try:
        checkpoint = torch.load(model_path, map_location=device)
        model.load_state_dict(checkpoint["model_state_dict"])
        model.eval()
    except Exception as e:
        print(f"Błąd ładowania modelu: {e}")
        return None

    features = []
    labels = []

    all_files = [(f, 0) for f in baseline_files] + [(f, 1) for f in position_files]

    for path, label in all_files:
        try:
            data = np.load(path)
            full_abp = data["abp"]
            full_cbfv = data["cbfv"]

            # Iteracja po segmentach
            for start_idx in range(0, min(len(full_abp), len(full_cbfv)), segment_len):
                abp_segment = full_abp[start_idx : start_idx + segment_len]
                cbfv_segment = full_cbfv[start_idx : start_idx + segment_len]

                if len(abp_segment) < 2000:
                    continue

                # Ekstrakcja z ABP
                kp_abp, desc_abp, _ = extract_sparse_representation(abp_segment, model, device)
                # Ekstrakcja z CBFV
                kp_cbfv, desc_cbfv, _ = extract_sparse_representation(cbfv_segment, model, device)

                kp_count_abp = len(kp_abp)
                kp_count_cbfv = len(kp_cbfv)

                # Zabezpieczenie przed brakiem punktów kluczowych
                if kp_count_abp == 0 or kp_count_cbfv == 0:
                    continue

                # DTW na deskryptorach (Sparse DTW)
                distance, path_dtw = fastdtw(desc_abp, desc_cbfv, dist=cosine)

                # Opóźnienie (Transit Time): średnia różnica w indeksach czasowych dopasowanych punktów
                delays = [kp_abp[i] - kp_cbfv[j] for i, j in path_dtw]
                avg_delay = np.mean(delays)

                # Nasz nowy wektor cech fizjologicznych
                feat = [kp_count_abp, kp_count_cbfv, avg_delay, distance]

                # Podział na 3 klasy: Baseline, Kucanie, Wstawanie
                segment_idx = start_idx // segment_len
                if label == 0:
                    fine_label = 0
                else:
                    fine_label = 1 if segment_idx % 2 == 0 else 2

                features.append(feat)
                labels.append(fine_label)

        except Exception as e:
            continue

    features = np.array(features)
    labels = np.array(labels)

    # Skalowanie - krytyczny element przy cechach o różnych rozpiętościach!
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)

    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    preds = kmeans.fit_predict(features_scaled)

    sil_score = silhouette_score(features_scaled, preds)
    ari_score = adjusted_rand_score(labels, preds)

    print(f"  -> Przeanalizowanych segmentów: {len(features)}")
    print(f"  -> Silhouette Score: {sil_score:.4f}")
    print(f"  -> Adjusted Rand Index (ARI): {ari_score:.4f}\n")

    return features_scaled, labels, preds, sil_score, ari_score, features


In [ ]:
# Uruchamiamy nowy eksperyment na najlepszym modelu po fine-tuningu, testując 3 klastry
print("Rozpoczynam ekstrakcję sygnałów i DTW dla k=3. Może to potrwać dłuższą chwilę...")
res_physio = evaluate_physio_clustering(
    best_model_path, baseline_files, position_files, device, n_clusters=3
)

if res_physio:
    features_scaled, labels, preds, sil, ari, raw_features = res_physio

    # --- 1. Wizualizacja klastrowania zredukowanego do 2D (PCA) ---
    pca = PCA(n_components=2, random_state=42)
    feats_2d = pca.fit_transform(features_scaled)

    plt.figure(figsize=(14, 6))

    # Wykres 1: Rzeczywiste etykiety (Baseline vs Kucanie vs Wstawanie)
    plt.subplot(1, 2, 1)
    scatter1 = plt.scatter(feats_2d[:, 0], feats_2d[:, 1], c=labels, cmap='Set1', alpha=0.8, edgecolor='k')
    plt.title("Real classes")
    plt.xlabel("PCA 1")
    plt.ylabel("PCA 2")
    handles1, _ = scatter1.legend_elements()
    plt.legend(handles1, ["Baseline", "Squat", "Standing up"], title="Rzeczywista Klasa")
    plt.grid(True, linestyle='--', alpha=0.6)

    # Wykres 2: Klastry K-Means (Próba znalezienia 3 stanów: Spoczynek, Kucanie, Wstawanie)
    plt.subplot(1, 2, 2)
    scatter2 = plt.scatter(feats_2d[:, 0], feats_2d[:, 1], c=preds, cmap='viridis', alpha=0.8, edgecolor='k')
    plt.title(f"Clusters K-Means (k=3)\nARI: {ari:.3f} | Sil: {sil:.3f}")
    plt.xlabel("PCA 1")
    plt.ylabel("PCA 2")
    handles2, _ = scatter2.legend_elements()
    plt.legend(handles2, [f"Cluster {i}" for i in range(3)], title="Predicted clusters")
    plt.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()

    # Przygotowanie DataFrame z surowymi cechami
    df_feats = pd.DataFrame(raw_features, columns=["KP_ABP", "KP_CBFV", "Avg_Delay_DTW", "DTW_Cost"])
    label_map = {0: "Baseline", 1: "Squat", 2: "Stand"}
    df_feats["Label"] = [label_map[l] for l in labels]
    df_feats["Cluster"] = [f"Klaster {p}" for p in preds]

    # --- 2. Wizualizacja relacji między konkretnymi cechami (Pairplot dla klas) ---
    print("\n--- Relacje między konkretnymi cechami (Rzeczywiste Klasy) ---")
    sns.pairplot(df_feats, hue="Label", vars=["KP_ABP", "KP_CBFV", "Avg_Delay_DTW", "DTW_Cost"], palette="Set1", diag_kind="kde", corner=True)
    plt.suptitle(f"Dependencies between signal characteristic features\nARI: {ari:.3f} | Sil: {sil:.3f}", y=1.02)
    plt.show()

    # --- 3. Wizualizacja rozkładu poszczególnych cech (Boxplot dla klas) ---
    plt.figure(figsize=(12, 8))
    for i, col in enumerate(["KP_ABP", "KP_CBFV", "Avg_Delay_DTW", "DTW_Cost"], 1):
        plt.subplot(2, 2, i)
        sns.boxplot(x="Label", y=col, hue="Label", data=df_feats, palette="Set1", legend=False)
        plt.title(f"Distribution of values for classes: {col}")
        plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()


In [ ]:
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Uruchamiamy eksperyment na najlepszym modelu po fine-tuningu, testując 2 klastry
print("Rozpoczynam ekstrakcję sygnałów i DTW dla k=2. Może to potrwać dłuższą chwilę...")
res_physio_2 = evaluate_physio_clustering(
    best_model_path, baseline_files, position_files, device, n_clusters=2
)

if res_physio_2:
    features_scaled, raw_labels_3, preds, sil, internal_ari, raw_features = res_physio_2

    # Przemapuj prawdziwe etykiety z 3 klas na 2 klasy (0: Baseline, 1: Zmiana Pozycji)
    labels_2 = np.where(raw_labels_3 > 0, 1, 0)

    # Przelicz ARI na poprawnych binarnych etykietach
    ari_2 = adjusted_rand_score(labels_2, preds)

    # --- 1. Wizualizacja klastrowania zredukowanego do 2D (PCA) ---
    pca = PCA(n_components=2, random_state=42)
    feats_2d = pca.fit_transform(features_scaled)

    plt.figure(figsize=(14, 6))

    # Wykres 1: Rzeczywiste etykiety (Baseline vs Zmiana pozycji)
    plt.subplot(1, 2, 1)
    scatter1 = plt.scatter(feats_2d[:, 0], feats_2d[:, 1], c=labels_2, cmap='Set1', alpha=0.8, edgecolor='k')
    plt.title("Real classes (2 classes)")
    plt.xlabel("PCA 1")
    plt.ylabel("PCA 2")
    handles1, _ = scatter1.legend_elements()
    plt.legend(handles1, ["Baseline", "Position Change"], title="Rzeczywista Klasa")
    plt.grid(True, linestyle='--', alpha=0.6)

    # Wykres 2: Klastry K-Means (Próba znalezienia 2 stanów: Baseline, Zmiana pozycji)
    plt.subplot(1, 2, 2)
    scatter2 = plt.scatter(feats_2d[:, 0], feats_2d[:, 1], c=preds, cmap='viridis', alpha=0.8, edgecolor='k')
    plt.title(f"Clusters K-Means (k=2)\nARI: {ari_2:.3f} | Sil: {sil:.3f}")
    plt.xlabel("PCA 1")
    plt.ylabel("PCA 2")
    handles2, _ = scatter2.legend_elements()
    plt.legend(handles2, [f"Cluster {i}" for i in range(2)], title="Predicted clusters")
    plt.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()

    # Przygotowanie DataFrame z surowymi cechami
    df_feats = pd.DataFrame(raw_features, columns=["KP_ABP", "KP_CBFV", "Avg_Delay_DTW", "DTW_Cost"])
    label_map = {0: "Baseline", 1: "Position Change"}
    df_feats["Label"] = [label_map[l] for l in labels_2]
    df_feats["Cluster"] = [f"Klaster {p}" for p in preds]

    # --- 2. Wizualizacja relacji między konkretnymi cechami (Pairplot dla klas) ---
    print("\n--- Relacje między konkretnymi cechami (Rzeczywiste Klasy) ---")
    sns.pairplot(df_feats, hue="Label", vars=["KP_ABP", "KP_CBFV", "Avg_Delay_DTW", "DTW_Cost"], palette="Set1", diag_kind="kde", corner=True)
    plt.suptitle(f"Dependencies between signal characteristic features\nARI: {ari_2:.3f} | Sil: {sil:.3f}", y=1.02)
    plt.show()

    # --- 3. Wizualizacja rozkładu poszczególnych cech (Boxplot dla klas) ---
    plt.figure(figsize=(12, 8))
    for i, col in enumerate(["KP_ABP", "KP_CBFV", "Avg_Delay_DTW", "DTW_Cost"], 1):
        plt.subplot(2, 2, i)
        sns.boxplot(x="Label", y=col, hue="Label", data=df_feats, palette="Set1", legend=False)
        plt.title(f"Distribution of values for classes: {col}")
        plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()
